# **Datasets Loader - FSC-147 & OmniCount-191**

**Note:**

**1. Dataset inspections are done in notebook Datasets_Inspection.ipynb which should be run first.**

**2. Datasets should have been downloaded and unzipped to their default locations (done by Datasets_Inspection.ipynb)**


In [1]:
# 0 - mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1 - Define Datasets paths

from pathlib import Path

# ============================================================
# Project / Dataset Paths
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

DATA_ROOT = PROJECT_ROOT / "datasets"


# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

FSC147_ROOT = DATA_ROOT / "FSC147"
FSC147_RAW = FSC147_ROOT / "raw"

FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
FSC147_DENSITY_DIR = FSC147_RAW / "gt_density_map_adaptive_384_VarV2"

FSC147_ANNOTATION_FILE = FSC147_RAW / "annotation_FSC147_384.json"
FSC147_SPLIT_FILE = FSC147_RAW / "Train_Test_Val_FSC_147.json"
FSC147_CLASSES_FILE = FSC147_RAW / "ImageClasses_FSC147.txt"


# ------------------------------------------------------------
# OmniCount-191
# ------------------------------------------------------------

OMNICOUNT_ROOT = DATA_ROOT / "OmniCount-191"

OMNICOUNT_DOMAINS = [
    "Birds",
    "Fruits",
    "Household",
    "Pets",
    "Satellite",
    "Supermarket",
    "Urban",
    "Vegetables",
    "Wild",
]

OMNICOUNT_SPLITS = [
    "train",
    "valid",
    "test",
]


# ------------------------------------------------------------
# Quick confirmation
# ------------------------------------------------------------

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)

print("\nFSC-147:")
print("FSC147_ROOT  :", FSC147_ROOT)
print("FSC147_RAW   :", FSC147_RAW)

print("\nOmniCount-191:")
print("OMNICOUNT_ROOT:", OMNICOUNT_ROOT)
print("Domains       :", OMNICOUNT_DOMAINS)

PROJECT_ROOT : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
DATA_ROOT    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets

FSC-147:
FSC147_ROOT  : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147
FSC147_RAW   : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147/raw

OmniCount-191:
OMNICOUNT_ROOT: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/OmniCount-191
Domains       : ['Birds', 'Fruits', 'Household', 'Pets', 'Satellite', 'Supermarket', 'Urban', 'Vegetables', 'Wild']


# Common Dataset Sample Schema

The FSC-147 and OmniCount-191 datasets have substantially different native
annotation structures.

The purpose of the dataloaders is therefore to convert both datasets into a
single common sample representation so that downstream pipeline components
(e.g. SAM2, semantic/VLM stages, geometric/depth stages, counting evaluation)
do not need dataset-specific branching.

The common schema returned by `__getitem__()` is:

```python
{
    "image_id": str,
    "image_path": str,
    "image": PIL.Image.Image | None,

    "width": int,
    "height": int,

    "categories": {
        "<class_name>": {
            "points": [[x, y], ...],
            "boxes": [[x1, y1, x2, y2], ...],
            "count": int
        },
        ...
    },

    "exemplar_boxes": [
        [x1, y1, x2, y2],
        ...
    ],

    "dataset": "fsc147" | "omnicount191",

    "domain": None | str,

    "split": "train" | "val" | "test"
}
```

---

## Field Definitions

### `image_id`

Dataset sample identifier returned as a string.

For FSC-147 this can be the original filename, e.g.:

```text
2.jpg
```

For OmniCount-191, native COCO `image_id` values are local to each
domain/split and are therefore not globally unique.

The OmniCount loader should therefore construct an unambiguous identifier
using domain/split context, for example:

```text
Birds/test/0
```

The original local COCO image ID may still be retained internally by the
loader.

---

### `image_path`

Full path to the corresponding image file.

---

### `image`

Loaded RGB `PIL.Image.Image`.

If:

```python
load_images=False
```

then:

```python
image = None
```

This allows metadata / annotation processing without repeatedly loading image
files from Google Drive.

---

### `width`, `height`

Dimensions of the image represented by the annotations.

These fields should remain available even when:

```python
load_images=False
```

For FSC-147 they can be obtained from the annotation metadata / resize ratios.

For OmniCount-191 they are supplied directly in the COCO `images` table.

---

## `categories`

Dictionary containing the target counting classes present in the sample.

Each semantic class has the same structure:

```python
{
    "points": [...],
    "boxes": [...],
    "count": ...
}
```

This allows FSC-147 and OmniCount-191 to expose a common interface even though
their native GT representations differ.

---

## FSC-147 Representation

Each FSC-147 image contains exactly one target counting class.

Primary GT:

```python
points
```

Therefore:

```python
categories[class_name]["points"] = GT_points
categories[class_name]["boxes"] = []
categories[class_name]["count"] = len(GT_points)
```

Example:

```python
{
    "sea shells": {
        "points": [
            [49.21, 243.83],
            [112.97, 240.53],
            ...
        ],
        "boxes": [],
        "count": 8
    }
}
```

FSC-147 does **not** provide complete per-instance GT bounding boxes.

Its `box_examples_coordinates` are exemplar boxes used to identify example
target objects and must therefore NOT be placed in:

```python
categories[class_name]["boxes"]
```

Instead, they are stored separately as:

```python
"exemplar_boxes"
```

For FSC-147:

```python
domain = None
```

---

## OmniCount-191 Representation

An OmniCount-191 image may contain multiple semantic counting classes.

Primary GT:

```python
COCO bounding boxes
```

Therefore, for each class:

```python
categories[class_name]["points"] = []
categories[class_name]["boxes"] = GT_instance_boxes
categories[class_name]["count"] = len(GT_instance_boxes)
```

Example:

```python
{
    "crow": {
        "points": [],
        "boxes": [
            [x1, y1, x2, y2],
            ...
        ],
        "count": 2
    },

    "pigeon": {
        "points": [],
        "boxes": [
            ...
        ],
        "count": 10
    },

    "raven": {
        "points": [],
        "boxes": [
            ...
        ],
        "count": 5
    }
}
```

OmniCount COCO bounding boxes are stored natively as:

```text
[x, y, width, height]
```

but the common loader will convert them to:

```text
[x1, y1, x2, y2]
```

The OmniCount loader will also clip very small sub-pixel boundary exceedances
to the image dimensions where required.

OmniCount does not use FSC-style exemplar boxes, therefore:

```python
exemplar_boxes = []
```

For OmniCount:

```python
domain = "Birds"
```

or another top-level OmniCount domain such as:

```text
Fruits
Household
Pets
Satellite
Supermarket
Urban
Vegetables
Wild
```

---

## Split Naming

The common external split names will be:

```text
train
val
test
```

FSC-147 already uses:

```text
train / val / test
```

OmniCount-191 stores its validation folders as:

```text
valid
```

The OmniCount loader should therefore map:

```text
val -> valid
```

internally.

This keeps downstream evaluation code dataset-independent.

---

## Design Principle

The meaning of the fields must remain consistent across datasets.

In particular:

```text
points
```

means complete GT object-location points where available.

```text
boxes
```

means complete per-instance GT object bounding boxes where available.

```text
exemplar_boxes
```

means example target-object regions and must not be confused with GT instance
boxes.

```text
count
```

is always the official dataset GT count for that semantic class.

This common representation allows downstream code to operate on:

```python
sample["categories"]
```

without needing to know whether the original source was FSC-147 or
OmniCount-191.

In [3]:
# ============================================================
# Step 2 - FSC-147 Dataset Loader
# ============================================================

import json
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset


class FSC147Dataset(Dataset):
    """
    FSC-147 loader using the common dataset schema.

    Expected FSC-147 structure:

        FSC147_RAW/
        ├── images_384_VarV2/
        ├── gt_density_map_adaptive_384_VarV2/
        ├── annotation_FSC147_384.json
        ├── Train_Test_Val_FSC_147.json
        └── ImageClasses_FSC147.txt

    Ground truth:
        - Complete object annotations are POINT annotations.
        - box_examples_coordinates contains exemplar boxes only.
        - Exemplar boxes are NOT complete per-instance GT boxes.
    """

    def __init__(self, root, split="train", load_images=True):

        # ----------------------------------------------------
        # Validate split
        # ----------------------------------------------------

        if split not in ("train", "val", "test"):
            raise ValueError(
                f"Invalid split '{split}'. "
                "Expected 'train', 'val', or 'test'."
            )

        self.root = Path(root)
        self.split = split
        self.load_images = load_images

        # ----------------------------------------------------
        # FSC-147 paths
        # ----------------------------------------------------

        self.image_dir = self.root / "images_384_VarV2"
        self.annotation_file = self.root / "annotation_FSC147_384.json"
        self.split_file = self.root / "Train_Test_Val_FSC_147.json"
        self.classes_file = self.root / "ImageClasses_FSC147.txt"

        # ----------------------------------------------------
        # Load annotations
        # ----------------------------------------------------

        with open(self.annotation_file, "r") as f:
            self.annotations = json.load(f)

        # ----------------------------------------------------
        # Load train / val / test split
        # ----------------------------------------------------

        with open(self.split_file, "r") as f:
            split_data = json.load(f)

        self.image_ids = split_data[self.split]

        # ----------------------------------------------------
        # Load image -> class mapping
        # ----------------------------------------------------

        self.image_to_class = {}

        with open(self.classes_file, "r") as f:
            for line in f:

                line = line.strip()

                if not line:
                    continue

                parts = line.split("\t")

                if len(parts) >= 2:
                    image_id = parts[0]
                    class_name = parts[1]

                    self.image_to_class[image_id] = class_name


    def __len__(self):

        return len(self.image_ids)


    def __getitem__(self, idx):

        # ----------------------------------------------------
        # Basic sample information
        # ----------------------------------------------------

        image_id = self.image_ids[idx]
        image_path = self.image_dir / image_id

        ann = self.annotations[image_id]

        class_name = self.image_to_class.get(
            image_id,
            "unknown"
        )

        # ----------------------------------------------------
        # Ground-truth points
        # ----------------------------------------------------

        points = ann.get("points", [])

        # ----------------------------------------------------
        # FSC-147 exemplar boxes
        #
        # These are NOT complete GT instance boxes.
        # Keep them separate from categories["boxes"].
        # ----------------------------------------------------

        exemplar_boxes_raw = ann.get(
            "box_examples_coordinates",
            []
        )

        exemplar_boxes = []

        for box in exemplar_boxes_raw:

            # FSC-147 exemplar coordinates are commonly stored
            # as four corner points:
            #
            # [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
            #
            # Convert them to [xmin, ymin, xmax, ymax].

            if len(box) == 4 and hasattr(box[0], "__len__"):

                xs = [p[0] for p in box]
                ys = [p[1] for p in box]

                exemplar_boxes.append([
                    min(xs),
                    min(ys),
                    max(xs),
                    max(ys),
                ])

        # ----------------------------------------------------
        # Image dimensions
        # ----------------------------------------------------

        width = None
        height = None
        image = None

        # FSC-147 annotation metadata contains dimensions
        # indirectly through H/W and resize ratios in the
        # standard release.

        if "W" in ann and "H" in ann:

            width = int(ann["W"])
            height = int(ann["H"])

        # If the image itself is requested, load it and use
        # its actual dimensions.

        if self.load_images:

            image = Image.open(image_path).convert("RGB")
            width, height = image.size

        # ----------------------------------------------------
        # Common sample schema
        # ----------------------------------------------------

        sample = {

            "image_id": str(image_id),

            "image_path": str(image_path),

            "image": image,

            "width": width,
            "height": height,

            "categories": {

                class_name: {

                    "points": points,

                    # FSC-147 has no complete GT instance boxes
                    "boxes": [],

                    "count": len(points),
                }
            },

            "exemplar_boxes": exemplar_boxes,

            "dataset": "fsc147",

            "domain": None,

            "split": self.split,
        }

        return sample

In [5]:
# ============================================================
# Step 2A - FSC-147 Loader Validation
# ============================================================

for split in ["train", "val", "test"]:

    dataset = FSC147Dataset(
        root=FSC147_RAW,
        split=split,
        load_images=False
    )

    print(f"\n=== FSC-147 {split.upper()} ===")
    print("Number of samples:", len(dataset))

    # Check first, middle and last samples
    indices = [0, len(dataset) // 2, len(dataset) - 1]

    for idx in indices:

        sample = dataset[idx]

        class_name = next(iter(sample["categories"]))
        category = sample["categories"][class_name]

        print(
            f"  idx={idx:<5} "
            f"image={sample['image_id']:<15} "
            f"class={class_name:<20} "
            f"count={category['count']:<4} "
            f"points={len(category['points']):<4} "
            f"exemplars={len(sample['exemplar_boxes'])}"
        )




=== FSC-147 TRAIN ===
Number of samples: 3659
  idx=0     image=7.jpg           class=peppers              count=13   points=13   exemplars=3
  idx=1829  image=4444.jpg        class=tomatoes             count=10   points=10   exemplars=3
  idx=3658  image=7362.jpg        class=boxes                count=105  points=105  exemplars=4

=== FSC-147 VAL ===
Number of samples: 1286
  idx=0     image=190.jpg         class=seagulls             count=13   points=13   exemplars=3
  idx=643   image=4571.jpg        class=flamingos            count=47   points=47   exemplars=3
  idx=1285  image=7049.jpg        class=birds                count=92   points=92   exemplars=4

=== FSC-147 TEST ===
Number of samples: 1190
  idx=0     image=2.jpg           class=sea shells           count=8    points=8    exemplars=3
  idx=595   image=4935.jpg        class=strawberries         count=9    points=9    exemplars=3
  idx=1189  image=6901.jpg        class=sheep                count=38   points=38   exemplars=

In [6]:
# ============================================================
# Step 3 - OmniCount-191 Dataset Loader
# ============================================================

from collections import defaultdict


class OmniCount191Dataset(Dataset):
    """
    OmniCount-191 loader using the common dataset schema.

    Actual downloaded structure:

        OMNICOUNT_ROOT/
        ├── Birds/
        │   ├── train/
        │   ├── valid/
        │   └── test/
        ├── Fruits/
        │   ├── train/
        │   ├── valid/
        │   └── test/
        ├── Household/
        │   └── train/
        └── ...

    Each available split directory contains:

        *.jpg
        _annotations.coco.json
        <domain>_vae_<split>.json

    Ground truth used by this loader:
        - COCO instance bounding boxes
        - category/class labels

    VAE JSON and sparse segmentation fields are not required for
    the common counting loader.
    """

    def __init__(
        self,
        root,
        split="train",
        domains=None,
        load_images=True
    ):

        # ----------------------------------------------------
        # Validate common split name
        # ----------------------------------------------------

        if split not in ("train", "val", "test"):
            raise ValueError(
                f"Invalid split '{split}'. "
                "Expected 'train', 'val', or 'test'."
            )

        self.root = Path(root)
        self.split = split
        self.load_images = load_images

        # OmniCount uses "valid" on disk,
        # while our common schema uses "val".
        self.disk_split = (
            "valid" if split == "val" else split
        )

        # ----------------------------------------------------
        # Select domains
        # ----------------------------------------------------

        if domains is None:
            self.domains = list(OMNICOUNT_DOMAINS)

        elif isinstance(domains, str):
            self.domains = [domains]

        else:
            self.domains = list(domains)

        # Validate requested domains
        unknown_domains = set(self.domains) - set(OMNICOUNT_DOMAINS)

        if unknown_domains:
            raise ValueError(
                f"Unknown OmniCount domain(s): {unknown_domains}"
            )

        # ----------------------------------------------------
        # Build dataset index
        #
        # One record = one image.
        # ----------------------------------------------------

        self.index = []

        for domain in self.domains:

            split_dir = (
                self.root
                / domain
                / self.disk_split
            )

            coco_path = (
                split_dir
                / "_annotations.coco.json"
            )

            # Some OmniCount domains do not contain
            # every train / valid / test split.
            if not coco_path.exists():
                continue

            # ------------------------------------------------
            # Load local COCO annotation file
            # ------------------------------------------------

            with open(coco_path, "r") as f:
                coco = json.load(f)

            # -----------------------------------------------
            # Local category lookup
            # -----------------------------------------------

            categories_by_id = {
                cat["id"]: cat["name"]
                for cat in coco["categories"]
            }

            # -----------------------------------------------
            # Group object annotations by local image_id
            # -----------------------------------------------

            annotations_by_image = defaultdict(list)

            for ann in coco["annotations"]:
                annotations_by_image[
                    ann["image_id"]
                ].append(ann)

            # -----------------------------------------------
            # Add each image to global loader index
            # -----------------------------------------------

            for img_info in coco["images"]:

                local_image_id = img_info["id"]

                self.index.append({
                    "domain": domain,
                    "split_dir": split_dir,
                    "image_info": img_info,
                    "annotations": annotations_by_image.get(
                        local_image_id,
                        []
                    ),
                    "categories_by_id": categories_by_id,
                })


    def __len__(self):

        return len(self.index)


    def __getitem__(self, idx):

        record = self.index[idx]

        domain = record["domain"]
        split_dir = record["split_dir"]

        img_info = record["image_info"]
        annotations = record["annotations"]
        categories_by_id = record["categories_by_id"]

        # ----------------------------------------------------
        # Local COCO image information
        # ----------------------------------------------------

        local_image_id = img_info["id"]
        file_name = img_info["file_name"]

        width = int(img_info["width"])
        height = int(img_info["height"])

        image_path = split_dir / file_name

        # Globally unambiguous identifier for common schema
        image_id = (
            f"{domain}/"
            f"{self.split}/"
            f"{local_image_id}"
        )

        # ----------------------------------------------------
        # Group GT boxes by semantic class
        # ----------------------------------------------------

        boxes_by_class = defaultdict(list)

        for ann in annotations:

            category_id = ann["category_id"]

            class_name = categories_by_id[
                category_id
            ]

            # COCO format:
            # [x, y, width, height]
            x, y, w, h = ann["bbox"]

            # Convert to common:
            # [x1, y1, x2, y2]
            x1 = float(x)
            y1 = float(y)
            x2 = float(x + w)
            y2 = float(y + h)

            # Clip to image boundaries.
            #
            # Our inspection found a very small number
            # of sub-pixel boundary exceedances.
            x1 = max(0.0, min(x1, float(width)))
            y1 = max(0.0, min(y1, float(height)))
            x2 = max(0.0, min(x2, float(width)))
            y2 = max(0.0, min(y2, float(height)))

            boxes_by_class[class_name].append([
                x1,
                y1,
                x2,
                y2
            ])

        # ----------------------------------------------------
        # Build common category structure
        # ----------------------------------------------------

        categories = {}

        for class_name, boxes in boxes_by_class.items():

            categories[class_name] = {
                "points": [],
                "boxes": boxes,
                "count": len(boxes),
            }

        # ----------------------------------------------------
        # Load image only if requested
        # ----------------------------------------------------

        image = None

        if self.load_images:

            image = Image.open(
                image_path
            ).convert("RGB")

            # Verify/use actual dimensions
            width, height = image.size

        # ----------------------------------------------------
        # Common sample schema
        # ----------------------------------------------------

        sample = {

            "image_id": image_id,

            "image_path": str(image_path),

            "image": image,

            "width": width,
            "height": height,

            "categories": categories,

            # OmniCount does not use FSC-style
            # exemplar boxes.
            "exemplar_boxes": [],

            "dataset": "omnicount191",

            "domain": domain,

            # Common external split naming:
            # train / val / test
            "split": self.split,
        }

        return sample

In [7]:
# ============================================================
# Step 3A - OmniCount-191 Loader Validation
# ============================================================

for split in ["train", "val", "test"]:

    dataset = OmniCount191Dataset(
        root=OMNICOUNT_ROOT,
        split=split,
        load_images=False
    )

    print(f"\n=== OmniCount-191 {split.upper()} ===")
    print("Number of samples:", len(dataset))

    # Check first, middle and last samples
    indices = [0, len(dataset) // 2, len(dataset) - 1]

    for idx in indices:

        sample = dataset[idx]

        print(
            f"\nidx={idx}"
            f" | image_id={sample['image_id']}"
            f" | domain={sample['domain']}"
            f" | split={sample['split']}"
            f" | size={sample['width']}x{sample['height']}"
        )

        print("  classes:", list(sample["categories"].keys()))

        total_count = 0

        for class_name, data in sample["categories"].items():

            total_count += data["count"]

            print(
                f"    {class_name:<20} "
                f"count={data['count']:<4} "
                f"points={len(data['points']):<4} "
                f"boxes={len(data['boxes']):<4}"
            )

        print("  total GT objects:", total_count)
        print("  exemplar boxes:", len(sample["exemplar_boxes"]))


=== OmniCount-191 TRAIN ===
Number of samples: 24704

idx=0 | image_id=Birds/train/0 | domain=Birds | split=train | size=1920x1080
  classes: ['raven', 'crow', 'pigeon']
    raven                count=4    points=0    boxes=4   
    crow                 count=2    points=0    boxes=2   
    pigeon               count=9    points=0    boxes=9   
  total GT objects: 15
  exemplar boxes: 0

idx=12352 | image_id=Urban/train/1652 | domain=Urban | split=train | size=640x640
  classes: ['car']
    car                  count=4    points=0    boxes=4   
  total GT objects: 4
  exemplar boxes: 0

idx=24703 | image_id=Wild/train/7002 | domain=Wild | split=train | size=640x640
  classes: ['wildboar']
    wildboar             count=5    points=0    boxes=5   
  total GT objects: 5
  exemplar boxes: 0

=== OmniCount-191 VAL ===
Number of samples: 3578

idx=0 | image_id=Birds/val/0 | domain=Birds | split=val | size=1920x1080
  classes: ['pigeon', 'raven', 'crow']
    pigeon               count=10   

In [8]:
# ============================================================
# Step 3B - Known OmniCount Sample Check
# ============================================================

birds_test = OmniCount191Dataset(
    root=OMNICOUNT_ROOT,
    split="test",
    domains="Birds",
    load_images=False
)

sample = birds_test[0]

print("image_id :", sample["image_id"])
print("domain   :", sample["domain"])
print("split    :", sample["split"])
print("size     :", sample["width"], "x", sample["height"])

print("\nGT classes:")

total = 0

for class_name, data in sorted(sample["categories"].items()):

    print(
        f"{class_name:<10} "
        f"count={data['count']} "
        f"boxes={len(data['boxes'])}"
    )

    total += data["count"]

print("\nTotal GT:", total)

image_id : Birds/test/0
domain   : Birds
split    : test
size     : 1920 x 1080

GT classes:
crow       count=2 boxes=2
pigeon     count=10 boxes=10
raven      count=5 boxes=5

Total GT: 17


In [9]:
# ============================================================
# Step 4 - Common PyTorch DataLoader
# ============================================================

from typing import List
from torch.utils.data import DataLoader


def collate_keep_dicts(batch: List[dict]) -> List[dict]:
    """
    Keep each sample as an individual dictionary.

    Counting samples contain:
        - variable numbers of classes
        - variable numbers of GT points
        - variable numbers of GT boxes

    Therefore, the default PyTorch collate function should not
    attempt to stack these ragged structures into tensors.

    Tensor conversion should be performed later by whichever
    pipeline stage requires it.
    """
    return batch

In [10]:
# ============================================================
# Step 4A - DataLoader Smoke Test
# ============================================================

# FSC-147
fsc_dataset = FSC147Dataset(
    root=FSC147_RAW,
    split="test",
    load_images=False
)

fsc_loader = DataLoader(
    fsc_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_keep_dicts
)

fsc_batch = next(iter(fsc_loader))


# OmniCount-191
omni_dataset = OmniCount191Dataset(
    root=OMNICOUNT_ROOT,
    split="test",
    load_images=False
)

omni_loader = DataLoader(
    omni_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_keep_dicts
)

omni_batch = next(iter(omni_loader))


# ------------------------------------------------------------
# Check batch structure
# ------------------------------------------------------------

print("=== FSC-147 BATCH ===")
print("Batch type:", type(fsc_batch))
print("Batch size:", len(fsc_batch))

for sample in fsc_batch:
    print(
        sample["image_id"],
        "| dataset =", sample["dataset"],
        "| domain =", sample["domain"],
        "| classes =", list(sample["categories"].keys())
    )


print("\n=== OmniCount-191 BATCH ===")
print("Batch type:", type(omni_batch))
print("Batch size:", len(omni_batch))

for sample in omni_batch:
    print(
        sample["image_id"],
        "| dataset =", sample["dataset"],
        "| domain =", sample["domain"],
        "| classes =", list(sample["categories"].keys())
    )

=== FSC-147 BATCH ===
Batch type: <class 'list'>
Batch size: 4
2.jpg | dataset = fsc147 | domain = None | classes = ['sea shells']
3.jpg | dataset = fsc147 | domain = None | classes = ['hot air balloons']
4.jpg | dataset = fsc147 | domain = None | classes = ['hot air balloons']
5.jpg | dataset = fsc147 | domain = None | classes = ['hot air balloons']

=== OmniCount-191 BATCH ===
Batch type: <class 'list'>
Batch size: 4
Birds/test/0 | dataset = omnicount191 | domain = Birds | classes = ['pigeon', 'crow', 'raven']
Birds/test/1 | dataset = omnicount191 | domain = Birds | classes = ['crow', 'pigeon', 'raven']
Birds/test/2 | dataset = omnicount191 | domain = Birds | classes = ['crow', 'pigeon', 'raven']
Birds/test/3 | dataset = omnicount191 | domain = Birds | classes = ['pigeon', 'crow', 'raven']


In [11]:
# ============================================================
# Step 4B - Image Loading Smoke Test
# ============================================================

from PIL import Image


# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

fsc_image_dataset = FSC147Dataset(
    root=FSC147_RAW,
    split="test",
    load_images=True
)

fsc_image_loader = DataLoader(
    fsc_image_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_keep_dicts
)

fsc_image_batch = next(iter(fsc_image_loader))


# ------------------------------------------------------------
# OmniCount-191
# ------------------------------------------------------------

omni_image_dataset = OmniCount191Dataset(
    root=OMNICOUNT_ROOT,
    split="test",
    load_images=True
)

omni_image_loader = DataLoader(
    omni_image_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_keep_dicts
)

omni_image_batch = next(iter(omni_image_loader))


# ------------------------------------------------------------
# Check loaded images
# ------------------------------------------------------------

print("=== FSC-147 IMAGE BATCH ===")

for sample in fsc_image_batch:

    image = sample["image"]

    print(
        f'{sample["image_id"]} | '
        f'type={type(image).__name__} | '
        f'mode={image.mode} | '
        f'size={image.size} | '
        f'stored_size={sample["width"]}x{sample["height"]}'
    )

    assert isinstance(image, Image.Image)
    assert image.mode == "RGB"
    assert image.size == (sample["width"], sample["height"])


print("\n=== OmniCount-191 IMAGE BATCH ===")

for sample in omni_image_batch:

    image = sample["image"]

    print(
        f'{sample["image_id"]} | '
        f'type={type(image).__name__} | '
        f'mode={image.mode} | '
        f'size={image.size} | '
        f'stored_size={sample["width"]}x{sample["height"]}'
    )

    assert isinstance(image, Image.Image)
    assert image.mode == "RGB"
    assert image.size == (sample["width"], sample["height"])


print("\nImage loading smoke test PASSED.")

=== FSC-147 IMAGE BATCH ===
2.jpg | type=Image | mode=RGB | size=(633, 384) | stored_size=633x384
3.jpg | type=Image | mode=RGB | size=(512, 384) | stored_size=512x384

=== OmniCount-191 IMAGE BATCH ===
Birds/test/0 | type=Image | mode=RGB | size=(1920, 1080) | stored_size=1920x1080
Birds/test/1 | type=Image | mode=RGB | size=(1920, 1080) | stored_size=1920x1080

Image loading smoke test PASSED.
